# Atelier séance 1 — Faire surapprendre un petit MLP

**Cours:** Master 2 Finance, Data et IA — Deep Learning  
**Repo:** [Amzil-AI/mlp-overfit-tabular](https://github.com/Amzil-AI/mlp-overfit-tabular)

Mission:
1. Régression (MSE) — California Housing  
2. Classification binaire (BCE) — Breast Cancer  
3. Afficher **train** et **validation** sur le même graphique

Pas de dropout, pas de weight decay. Le but est de **voir l'écart grandir**.

## 1. Install (Colab)

In [ ]:
# Torch is preinstalled on Colab. This covers a fresh runtime.
%pip install -q scikit-learn matplotlib

## 2. Train + plot (full lab)

In [ ]:
from __future__ import annotations

import json
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import torch
import torch.nn as nn
from sklearn.datasets import fetch_california_housing, load_breast_cancer
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from torch.utils.data import DataLoader, TensorDataset

SEED = 42
BATCH_SIZE = 64
EPOCHS = 120
LR = 1e-3
HIDDEN = (256, 256, 128)  # wide on purpose → easier to memorize train
VAL_SIZE = 0.25

np.random.seed(SEED)
torch.manual_seed(SEED)

FIG_DIR = Path("figures")
OUT_DIR = Path("outputs")
FIG_DIR.mkdir(exist_ok=True)
OUT_DIR.mkdir(exist_ok=True)


class MLP(nn.Module):
    def __init__(self, n_in: int, n_out: int = 1):
        super().__init__()
        layers = []
        prev = n_in
        for h in HIDDEN:
            layers += [nn.Linear(prev, h), nn.ReLU()]
            prev = h
        layers.append(nn.Linear(prev, n_out))
        self.net = nn.Sequential(*layers)

    def forward(self, x):
        return self.net(x)


def make_loaders(X, y, *, classification):
    X_tr, X_va, y_tr, y_va = train_test_split(
        X, y, test_size=VAL_SIZE, random_state=SEED,
        stratify=y if classification else None,
    )
    scaler = StandardScaler().fit(X_tr)
    X_tr = scaler.transform(X_tr).astype(np.float32)
    X_va = scaler.transform(X_va).astype(np.float32)
    y_tr = y_tr.astype(np.float32).reshape(-1, 1)
    y_va = y_va.astype(np.float32).reshape(-1, 1)
    train_loader = DataLoader(
        TensorDataset(torch.from_numpy(X_tr), torch.from_numpy(y_tr)),
        batch_size=BATCH_SIZE, shuffle=True,
    )
    val_loader = DataLoader(
        TensorDataset(torch.from_numpy(X_va), torch.from_numpy(y_va)),
        batch_size=BATCH_SIZE, shuffle=False,
    )
    return train_loader, val_loader, X_tr.shape[1]


@torch.no_grad()
def epoch_loss(model, loader, criterion):
    model.eval()
    total, n = 0.0, 0
    for xb, yb in loader:
        loss = criterion(model(xb), yb)
        total += float(loss.item()) * len(xb)
        n += len(xb)
    return total / max(n, 1)


def train_task(name, X, y, *, classification):
    train_loader, val_loader, n_in = make_loaders(X, y, classification=classification)
    model = MLP(n_in)
    criterion = nn.BCEWithLogitsLoss() if classification else nn.MSELoss()
    opt = torch.optim.Adam(model.parameters(), lr=LR)  # no weight decay

    train_hist, val_hist = [], []
    for _ in range(EPOCHS):
        model.train()
        for xb, yb in train_loader:
            opt.zero_grad()
            loss = criterion(model(xb), yb)
            loss.backward()
            opt.step()
        train_hist.append(epoch_loss(model, train_loader, criterion))
        val_hist.append(epoch_loss(model, val_loader, criterion))

    gap = [v - t for t, v in zip(train_hist, val_hist)]
    diverge = None
    for i in range(5, len(gap)):
        if gap[i] > max(0.02 * (val_hist[0] + 1e-8), 1e-4) and gap[i] > gap[i - 5]:
            diverge = i + 1
            break

    return {
        "name": name,
        "loss": "BCEWithLogits" if classification else "MSE",
        "n_train": len(train_loader.dataset),
        "n_val": len(val_loader.dataset),
        "train_loss_final": train_hist[-1],
        "val_loss_final": val_hist[-1],
        "gap_final": gap[-1],
        "diverge_epoch_approx": diverge,
        "train_hist": train_hist,
        "val_hist": val_hist,
    }


def plot_curves(result):
    epochs = np.arange(1, len(result["train_hist"]) + 1)
    fig, ax = plt.subplots(figsize=(8, 4.5))
    ax.plot(epochs, result["train_hist"], label="Train", color="#1f4e79", lw=2)
    ax.plot(epochs, result["val_hist"], label="Validation", color="#c0392b", lw=2)
    if result["diverge_epoch_approx"]:
        ax.axvline(
            result["diverge_epoch_approx"], color="#7f8c8d", ls="--", lw=1.2,
            label=f"Écart visible ≈ époque {result['diverge_epoch_approx']}",
        )
    ax.set_xlabel("Époque")
    ax.set_ylabel(result["loss"])
    ax.set_title(f"{result['name']} — train vs validation")
    ax.legend(frameon=False)
    ax.grid(True, alpha=0.25)
    fig.tight_layout()
    return fig


housing = fetch_california_housing()
rng = np.random.default_rng(SEED)
idx = rng.choice(len(housing.data), size=2500, replace=False)
reg = train_task(
    "California Housing (régression)",
    housing.data[idx], housing.target[idx], classification=False,
)

cancer = load_breast_cancer()
clf = train_task(
    "Breast Cancer (classification binaire)",
    cancer.data, cancer.target.astype(np.float32), classification=True,
)

fig_reg = plot_curves(reg)
fig_reg.savefig(FIG_DIR / "regression_curves.png", dpi=140)
plt.show()

fig_clf = plot_curves(clf)
fig_clf.savefig(FIG_DIR / "classification_curves.png", dpi=140)
plt.show()

summary = {
    "architecture": {"hidden": list(HIDDEN), "activation": "ReLU", "regularization": "none"},
    "regression": {k: v for k, v in reg.items() if k not in ("train_hist", "val_hist")},
    "classification": {k: v for k, v in clf.items() if k not in ("train_hist", "val_hist")},
}
(OUT_DIR / "metrics.json").write_text(json.dumps(summary, indent=2))

print("Regression   train/val:", round(reg["train_loss_final"], 4), round(reg["val_loss_final"], 4),
      "| gap starts ~", reg["diverge_epoch_approx"])
print("Classification train/val:", round(clf["train_loss_final"], 6), round(clf["val_loss_final"], 4),
      "| gap starts ~", clf["diverge_epoch_approx"])

## 3. What to write in your livrable

Look at both charts and answer:

- Does the **train / validation gap** grow?
- **From which epoch** does it become clear?

Expected pattern: train keeps falling; validation flattens or rises. That is intentional overfitting — useful for séance 2 (dropout / learning rate).